
# RQ4 — το φίλτρο σχεδόν ολικής κοπής (F1)

**Τι κλείνει.** Το F3 έδειξε ότι το μερικό φιλτράρισμα δεν σώζει τη συνεργασία
σε κανένα μοντέλο. Η διατύπωση «είναι χειρότερο και από το ολικό μπλοκάρισμα»
όμως δεν στέκει ακόμα αυστηρά: η μόνη μας αναφορά ολικού μπλοκαρίσματος είναι
το σενάριο `silence`, που **δεν έχει ανταγωνιστικό πλαίσιο** — άρα η σύγκριση
μπερδεύει το φίλτρο με το πλαίσιο.

Το **F1** κόβει 93,2 % των μηνυμάτων στο ανταγωνιστικό κελί, έναντι 55,5 % του
F3. Είναι ουσιαστικά ολικό μπλοκάρισμα **με το εχθρικό πλαίσιο παρόν**, που
είναι ακριβώς η αναφορά που λείπει.

| έκβαση | συμπέρασμα |
|---|---|
| F1 > F3 | το μερικό φιλτράρισμα είναι χειρότερο από το ολικό — αποδεδειγμένο |
| F1 ≈ F3 | το φιλτράρισμα δεν δουλεύει σε καμία ένταση |
| F1 < F3 | απροσδόκητο· το κόψιμο των πάντων βλάπτει περισσότερο από το μισό |

Και οι τρεις είναι πληροφοριακές. Δεν υπάρχει κενή έκβαση.

**Τι δεν αλλάζει.** Ίδιο παιχνίδι, ίδιες αποδόσεις, ίδιοι 16 γύροι, ίδιο
`max_tokens` ανά μοντέλο με το αφιλτράριστο και με το F3 — ώστε η τριπλή
σύγκριση να μην κουβαλάει κανένα επιπλέον confound. Τα υπάρχοντα 1.420 runs
δεν αγγίζονται· η έξοδος πάει σε δικό της δέντρο.



## Setup

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN`, **Attach to notebook**
   *(χρειάζεται για gemma ×2 και Llama· τα Qwen όχι)*

Μετά: **Save Version → Save & Run All**.


In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# Το repo root ΕΙΝΑΙ το package: campaign.py και τα υπόλοιπα κάθονται στην
# κορυφή του clone, όχι σε υποφάκελο.
assert os.path.exists('campaign.py'), f'campaign.py δεν είναι στο {os.getcwd()}'
# Χωρίς αυτά τα δύο ορίσματα το notebook θα έτρεχε σιωπηλά λάθος πείραμα --
# έχει ξανασυμβεί.
src = open('campaign.py').read()
assert '--message-filter' in src and '--games' in src, \
    'Το repo δεν έχει --message-filter/--games -- κάνε git push πρώτα.'

print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    print('Χωρίς HF_TOKEN (εντάξει μόνο για Qwen):', e)


## Τι θα τρέξει

Και τα πέντε μοντέλα σε αυτή τη συνεδρία, **από το γρηγορότερο προς το
αργότερο**: αν χτυπήσει το 12ωρο, θα έχουν προλάβει όσο περισσότερα γίνεται,
και κάθε μοντέλο ζιπάρεται μόλις τελειώσει.

Εκτίμηση από τους μετρημένους ρυθμούς: 640 κλήσεις ανά μοντέλο, σύνολο
**περίπου 7–8 ώρες**. Για παράλληλες συνεδρίες, κόψε τη λίστα `MODELS` σε δύο
notebooks.


In [ ]:
# Γρηγορότερο πρώτα (μετρημένα s/call: Qwen2.5 ~5, Llama ~9.4,
# gemma-2-9b ~10.7, Qwen3-4B ~11). Το gemma-2-2b δεν έχει μετρηθεί ποτέ αλλά
# είναι το μικρότερο.
MODELS = [
    'google/gemma-2-2b-it',
    'Qwen/Qwen2.5-7B-Instruct',
    'meta-llama/Llama-3.1-8B-Instruct',
    'google/gemma-2-9b-it',
    'Qwen/Qwen3-4B',
]

SESSION        = 'B'
TOPOLOGY       = 'star'
SCENARIOS      = ['framing_competitive']
GAMES          = ['pd']
MESSAGE_FILTER = 'F1_competitive'

# Σταμάτα πριν ξεκινήσεις μοντέλο που δεν προλαβαίνει, αντί να σε κόψει το
# Kaggle στη μέση και να χαθεί η μισή δουλειά του.
WALL_HOURS = 11.0

def args_for(model):
    a = ['--model', model, '--session', SESSION, '--topology', TOPOLOGY,
         '--scenarios', *SCENARIOS, '--games', *GAMES]
    if MESSAGE_FILTER != 'none':
        a += ['--message-filter', MESSAGE_FILTER]
    return a

print(f'{len(MODELS)} μοντέλα, φίλτρο {MESSAGE_FILTER}, {SCENARIOS} {GAMES} {TOPOLOGY}')


## Preview — κοστίζει μηδέν

Δεν φορτώνει μοντέλο. Ελέγχει ότι κάθε πλάνο είναι σωστό **πριν** ξοδευτεί ώρα
GPU. Πρέπει να δεις για κάθε μοντέλο `expecting : 5 run files`, γραμμή
`FILTER : F1_competitive`, και `games : ['pd']`.

Αν λείπει η γραμμή FILTER ή το expecting λέει 10, κάτι δεν πέρασε — σταμάτα εδώ.


In [ ]:
import subprocess, sys

ok = True
for m in MODELS:
    print('=' * 70)
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(m), '--dry-run'],
                       capture_output=True, text=True)
    print(r.stdout.strip()[:900])
    if r.returncode != 0:
        ok = False
        print('ΣΦΑΛΜΑ:', r.stderr.strip()[-300:])

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')


## Εκτέλεση

Ένα μοντέλο τη φορά. Αν κάποιο αποτύχει, **η σειρά συνεχίζει** — δεν θέλουμε
μια αποτυχία στο τρίτο μοντέλο να ακυρώσει τα δύο πρώτα που ήδη πέτυχαν. Ο
απολογισμός στο τέλος λέει τι πέρασε.


In [ ]:
import subprocess, sys, time, os

t0 = time.time()
results = []

for m in MODELS:
    elapsed_h = (time.time() - t0) / 3600
    if elapsed_h > WALL_HOURS:
        print(f'\n[STOP] {elapsed_h:.1f} h -- δεν ξεκινάω το {m}, δεν προλαβαίνει.')
        results.append((m, 'skipped', 0.0))
        continue

    print('\n' + '=' * 70)
    print(f'{m}   ({elapsed_h:.1f} h μέχρι τώρα)')
    print('=' * 70, flush=True)

    t = time.time()
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(m)])
    mins = (time.time() - t) / 60
    status = 'OK' if r.returncode == 0 else f'FAILED (exit {r.returncode})'
    results.append((m, status, mins))
    print(f'\n--> {m}: {status}, {mins:.0f} λεπτά', flush=True)

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for m, s, mins in results:
    print(f'  {s:22s} {mins:6.0f} λ   {m}')
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h')


## Μάζεμα


In [ ]:
import glob, os, shutil

zips = sorted(glob.glob('/kaggle/working/*F1_competitive*.zip'))
print(f'{len(zips)} zip ανά μοντέλο:')
for z in zips:
    print(f'   {os.path.getsize(z)/1e6:6.1f} MB  {os.path.basename(z)}')

if zips:
    box = '/kaggle/working/f1_all'
    os.makedirs(box, exist_ok=True)
    for z in zips:
        shutil.copy(z, box)
    out = shutil.make_archive('/kaggle/working/f1_runs_all', 'zip', box)
    print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
else:
    print('\nΚΑΝΕΝΑ ZIP -- δες τον απολογισμό παραπάνω.')


## Μετά

Κατέβασε το `f1_runs_all.zip` στο `diplomatikh/rq4/filtered_F1/`.

Η σύγκριση που θα κάνουμε είναι τριπλή, ανά μοντέλο, στο ίδιο κελί:

| | ποσοστό κοπής |
|---|---|
| χωρίς φίλτρο (n=10) | 0 % |
| **F3** | 55,5 % |
| **F1** | 93,2 % |

Αν η συνεργασία ανεβαίνει μονότονα με το ποσοστό κοπής, τότε το μερικό
φιλτράρισμα είναι το χειρότερο σημείο της καμπύλης — και ο μηχανισμός που το
εξηγεί, ότι το φίλτρο συμπυκνώνει τα ψέματα στο κανάλι που επιβιώνει, γίνεται
ισχυρισμός με τρία σημεία αντί για δύο.

Αν η συνεδρία κοπεί, τα ανά-μοντέλο zip στο Output panel είναι ήδη καλά —
πάρ' τα πριν κλείσεις.
